<a href="https://colab.research.google.com/github/buihoangviet2982006-dotcom/email_spam_classifier/blob/feature%2Fdecision-tree/SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Đọc tập dữ liệu trên Google Drive

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay,
)

# Kết nối Google Drive
drive.mount('/content/drive')

# Đường dẫn thư mục chứa train.csv và test.csv
data_dir = '/content/drive/MyDrive/Colab Notebooks/Đồ án máy học/data/processed'
train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))

# Chuẩn bị dữ liệu
X_train = train_df['text'].fillna('').astype(str)
y_train = train_df['label']
X_test = test_df['text'].fillna('').astype(str)
y_test = test_df['label']

print("Đã tải dữ liệu thành công!")
print(f"Train: {len(X_train):,} mẫu | Test: {len(X_test):,} mẫu")

2. Huấn luyện mô hình (Baseline SVM)

In [ ]:
baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1, 2))),
    ('clf', LinearSVC(dual=True, max_iter=2000, random_state=42))
])

start_base = time.time()
baseline_pipeline.fit(X_train, y_train)
base_time = time.time() - start_base

y_pred_base = baseline_pipeline.predict(X_test)
y_score_base = baseline_pipeline.decision_function(X_test)

print("=== KẾT QUẢ BASELINE: SVM (MẶC ĐỊNH) ===")
print(f"Train Time : {base_time:.2f} giây")
print(f"Accuracy   : {accuracy_score(y_test, y_pred_base):.4f}")
print(f"F1-Score   : {f1_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC    : {roc_auc_score(y_test, y_score_base):.4f}")

3. Tinh chỉnh siêu tham số (GridSearchCV)

In [ ]:
param_grid = {
    'clf__C': [0.1, 0.5, 1.0, 5.0],
    'clf__loss': ['hinge', 'squared_hinge']
}

grid_search = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

start_tune = time.time()
grid_search.fit(X_train, y_train)
tune_time = time.time() - start_tune

best_model = grid_search.best_estimator_

print(f"\nTham số tốt nhất: {grid_search.best_params_}")
print(f"F1-Score trung bình (5-Fold CV): {grid_search.best_score_:.4f}")
print(f"Tổng thời gian chạy GridSearchCV: {tune_time:.2f} giây")

4. Chỉ số thực nghiệm

In [ ]:
# Đo thời gian huấn luyện riêng của mô hình tốt nhất
start_fit = time.time()
best_model.fit(X_train, y_train)
best_train_time = time.time() - start_fit

# Đo thời gian dự đoán trên tập Test
start_pred = time.time()
y_pred = best_model.predict(X_test)
y_score = best_model.decision_function(X_test)
predict_time = time.time() - start_pred

print("=== KẾT QUẢ THỰC NGHIỆM: SVM (TUNED) ===")
print(f"Best Params      : {grid_search.best_params_}")
print(f"Accuracy         : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (Spam) : {precision_score(y_test, y_pred):.4f}")
print(f"Recall (Spam)    : {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score (Spam)  : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC          : {roc_auc_score(y_test, y_score):.4f}")
print(f"Train Time       : {best_train_time:.2f} giây")
print(f"Predict Time     : {predict_time:.2f} giây")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=['Ham (0)', 'Spam (1)'], digits=4))

5. Biểu đồ Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham (0)', 'Spam (1)'])
disp.plot(cmap='Blues', values_format='d', ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix - SVM (LinearSVC)')

# 2. ROC Curve
RocCurveDisplay.from_predictions(y_test, y_score, name='SVM (LinearSVC)', ax=axes[1], color='#DD8452')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Guess')
axes[1].set_title('ROC Curve - SVM (LinearSVC)')
axes[1].legend()

plt.tight_layout()
plt.show()

6. Kiểm tra các từ đặc trưng & Phân tích lỗi

In [ ]:
tfidf = best_model.named_steps['tfidf']
clf = best_model.named_steps['clf']
feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_[0]

print("Top 10 từ/cụm từ đặc trưng cho SPAM:")
for idx in np.argsort(coefs)[-10:][::-1]:
    print(f"  {feature_names[idx]:<25} : +{coefs[idx]:.4f}")

print("\nTop 10 từ/cụm từ đặc trưng cho HAM:")
for idx in np.argsort(coefs)[:10]:
    print(f"  {feature_names[idx]:<25} : {coefs[idx]:.4f}")

# Xem 5 email dự đoán sai
error_df = pd.DataFrame({
    'text': X_test,
    'true_label': y_test,
    'pred_label': y_pred,
    'decision_score': y_score.round(4)
})
errors = error_df[error_df['true_label'] != error_df['pred_label']]
print(f"\nTổng số email đoán sai: {len(errors):,} / {len(y_test):,} ({len(errors)/len(y_test)*100:.2f}%)")
display(errors.head(5))

7. Test, hiệu chỉnh xác suất và lưu lại mô hình

Cell 7a — Hiệu chỉnh xác suất (để lấy % độ tin cậy) và lưu file

In [ ]:
import pickle

# Tạo pipeline SVM có hiệu chỉnh xác suất (CalibratedClassifierCV) từ tham số tốt nhất
best_C = grid_search.best_params_['clf__C']
best_loss = grid_search.best_params_['clf__loss']

calibrated_svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1, 2))),
    ('clf', CalibratedClassifierCV(
        estimator=LinearSVC(C=best_C, loss=best_loss, dual=True, max_iter=2000, random_state=42),
        cv=3
    ))
])

print("Đang hiệu chỉnh xác suất cho mô hình SVM...")
calibrated_svm_pipeline.fit(X_train, y_train)

# 1. Test thử dự đoán 2 email mẫu
def predict_email(text):
    prob = calibrated_svm_pipeline.predict_proba([text])[0][1]
    label = "SPAM" if prob >= 0.5 else "HAM"
    print(f"[{label}] (Xác suất Spam: {prob*100:.2f}%) -> {text[:60]}...")

print("\n--- DEMO DỰ ĐOÁN ---")
predict_email("Congratulations! You won a free $1000 gift card. Click here to claim your prize now!")
predict_email("Hi team, please find attached the meeting notes from yesterday's project sync.")

# 2. Lưu file mô hình dạng .pkl trực tiếp vào thư mục gốc của Colab (/content/)
model_path = '/content/svm_spam_pipeline.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(calibrated_svm_pipeline, f)

print(f"\nĐã lưu mô hình thành công tại: {model_path}")
print("-> Mở biểu tượng Thư mục (Files) bên thanh trái Colab, tìm file .pkl và bấm dấu 3 chấm -> Tải xuống (Download).")

Cell 7b — Giao diện kiểm tra trực tiếp trên Colab

In [ ]:
import re
import pickle
import ipywidgets as widgets
from IPython.display import display, clear_output

# Nạp mô hình (dùng trực tiếp trong RAM nếu có, nếu không thì đọc từ file .pkl)
try:
    model_to_test = calibrated_svm_pipeline
except NameError:
    with open('/content/svm_spam_pipeline.pkl', 'rb') as f:
        model_to_test = pickle.load(f)

# Tạo giao diện ô nhập liệu và nút bấm
text_input = widgets.Textarea(
    value='',
    placeholder='Dán hoặc nhập nội dung email tiếng Anh cần kiểm tra vào đây...',
    description='Nội dung:',
    layout=widgets.Layout(width='80%', height='150px')
)

predict_btn = widgets.Button(
    description='Kiểm tra Email',
    button_style='primary',
    icon='search'
)

clear_btn = widgets.Button(
    description='Xóa nội dung',
    button_style='',
    icon='trash'
)

output_area = widgets.Output()

def on_predict_clicked(b):
    with output_area:
        clear_output()
        raw_text = text_input.value.strip()
        if not raw_text:
            print("⚠️ Vui lòng nhập nội dung email trước khi kiểm tra!")
            return

        # Chuẩn hóa giống bước preprocessing
        cleaned = re.sub(r'\s+', ' ', raw_text.lower()).strip()

        # Dự đoán
        prob_spam = model_to_test.predict_proba([cleaned])[0][1]
        prob_ham = 1 - prob_spam

        if prob_spam >= 0.5:
            print("🚨 KẾT QUẢ: SPAM (Thư rác)")
            print(f"   Độ tin cậy: {prob_spam * 100:.2f}%")
        else:
            print("✅ KẾT QUẢ: HAM (Email bình thường)")
            print(f"   Độ tin cậy: {prob_ham * 100:.2f}%")

def on_clear_clicked(b):
    text_input.value = ''
    with output_area:
        clear_output()

predict_btn.on_click(on_predict_clicked)
clear_btn.on_click(on_clear_clicked)

display(text_input, widgets.HBox([predict_btn, clear_btn]), output_area)